# Building Age Classification from Satellite Imagery

# Notebook 4 — Spatial-Temporal Modelling & Evaluation (Baseline 60-Feature)

---

**Approach: Spatial-Temporal Gradient Boosting**

In this notebook, we transition from the original baseline approach to a **Spatial-Temporal LightGBM Engine**. While we are using the original 60-feature dataset (without advanced physical indices), we improve the modeling architecture significantly:
1. **Spatial Neighborhoods (KD-Trees):** For every pixel, we calculate the mean and standard deviation of its neighbors at scales of 16, 64, and 256 pixels to capture block-level urban context.
2. **Fold-Safe Cross-Validation:** To prevent data leakage, spatial neighborhoods are generated *inside* the cross-validation loop. Validation pixels only draw spatial context from the training set.

## 4.1 Setup & Imports

We import the necessary libraries for our upgraded **Spatial-Temporal LightGBM Engine**. We replace the older random forest imports with `LGBMClassifier` and include `cKDTree` for spatial neighborhood calculations and `GroupKFold` for leak-free spatial cross-validation.

In [ ]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from time import perf_counter

from scipy.spatial import cKDTree
from lightgbm import LGBMClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

CLASS_COLORS = {1: '#4393c3', 2: '#2ca25f', 3: '#fd8d3c', 4: '#de2d26'}
CLASS_LABELS_MADRID    = {1: 'Pre-1960',  2: '1960-1984', 3: '1984-2004', 4: '2004-2024'}
CLASS_LABELS_AMSTERDAM = {1: 'Pre-1945',  2: '1945-1984', 3: '1984-2004', 4: '2004-2024'}

# ── Configuration Settings ───────────────────────────────────────────────────
RANDOM_STATE = 42
N_SPLITS = 10
SPATIAL_BLOCK_SIZE = 10
NEIGHBOR_SCALES = [16, 64, 256]
KNN_BATCH_SIZE = 5000
KNN_WORKERS = -1

LGBM_PARAMS = dict(
    objective="multiclass",
    n_estimators=700,
    learning_rate=0.035,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=25,
    subsample=0.90,
    colsample_bytree=0.90,
    reg_alpha=0.15,
    reg_lambda=1.5,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)

plt.rcParams['figure.dpi'] = 100
print("Setup complete. Spatial LightGBM environment ready.")

## 4.2 Load Preprocessed Data

We load the original `preprocessed_data.pkl` file, extracting both Madrid and Amsterdam arrays along with their geographic pixel coordinates (`pixel_ids_madrid` and `pixel_ids_amsterdam`) which contain the `px_key` and `py_key` values required for spatial neighborhood queries.## 4.2 Load Preprocessed Data

In [ ]:
DATA_PATH = Path('./data/preprocessed/preprocessed_data.pkl')
print(f"Loading preprocessed data from:\n  {DATA_PATH}")

with open(DATA_PATH, 'rb') as f:
    data = pickle.load(f)

X_madrid        = np.asarray(data['X_madrid'], dtype=np.float64)
y_madrid        = np.asarray(data['y_madrid'], dtype=int)
X_amsterdam     = np.asarray(data['X_amsterdam'], dtype=np.float64)
y_amsterdam     = np.asarray(data['y_amsterdam'], dtype=int)
feature_names   = list(data['feature_names'])

coords_madrid    = np.asarray(data['pixel_ids_madrid'], dtype=np.float64)
coords_amsterdam = np.asarray(data['pixel_ids_amsterdam'], dtype=np.float64)

print(f'Madrid:    {X_madrid.shape}  ({len(np.unique(y_madrid))} classes)')
print(f'Amsterdam: {X_amsterdam.shape}  ({len(np.unique(y_amsterdam))} classes)')
print(f'Features:  {len(feature_names)}')
print()

# Print class distributions for both cities
for name, y, labels in [('Madrid', y_madrid, CLASS_LABELS_MADRID),
                         ('Amsterdam', y_amsterdam, CLASS_LABELS_AMSTERDAM)]:
    print(f'{name} class distribution:')
    for cls in sorted(np.unique(y)):
        n = (y == cls).sum()
        print(f'  Class {cls} ({labels[cls]}): {n:,}  ({n/len(y)*100:.1f}%)')
    print()

# Build working DataFrames with coordinate keys for spatial operations
df_madrid = pd.DataFrame(X_madrid, columns=feature_names)
df_madrid["px_key"] = coords_madrid[:, 0]
df_madrid["py_key"] = coords_madrid[:, 1]
df_madrid["age_class"] = y_madrid

df_amsterdam = pd.DataFrame(X_amsterdam, columns=feature_names)
df_amsterdam["px_key"] = coords_amsterdam[:, 0]
df_amsterdam["py_key"] = coords_amsterdam[:, 1]
df_amsterdam["age_class"] = y_amsterdam

---

## 4.3 Spatial Cross-Validation on Madrid

To leverage spatial context without leaking validation labels, we implement a **fold-safe KD-Tree neighborhood engine**. 
1. **Source Features:** We pick 12 strong base features (spectral bands, standard deviations, and early/late signals) to summarize local neighborhoods.
2. **Multi-Scale Neighborhoods:** We compute local means and standard deviations across scales of 16, 64, and 256 neighboring pixels using batched `cKDTree` queries.
3. **Spatial Block Groups:** We partition the map using coordinate blocks (`SPATIAL_BLOCK_SIZE = 10`) and evaluate performance using a 10-fold `GroupKFold`.

In [ ]:
def choose_neighbour_source_features(features):
    preferred = [
        "NIR_mean", "Red_mean", "SWIR1_mean", "NDVI_mean",
        "NDBI_mean", "MNDWI_mean", "NIR_std", "Red_std",
        "NIR_early_mean", "Red_late_std", "SWIR1_late_std", "d_NIR_mean",
    ]
    chosen = [f for f in preferred if f in features]
    if len(chosen) < 8:
        for f in features:
            if f.startswith("has_"): continue
            if f not in chosen: chosen.append(f)
            if len(chosen) >= 12: break
    return chosen[:12]

def add_neighbourhood_features(query_df, reference_df, source_features, scales=NEIGHBOR_SCALES, exclude_self=False, batch_size=KNN_BATCH_SIZE):
    if len(reference_df) == 0:
        raise ValueError("reference_df is empty")

    scales = sorted(scales)
    max_k = max(scales)

    query_xy = query_df[["px_key", "py_key"]].to_numpy(dtype=np.float64)
    reference_xy = reference_df[["px_key", "py_key"]].to_numpy(dtype=np.float64)
    reference_values = reference_df[source_features].to_numpy(dtype=np.float64)

    tree = cKDTree(reference_xy)
    requested_k = max_k + 1 if exclude_self else max_k
    requested_k = min(requested_k, len(reference_df))

    n_query = len(query_df)
    n_source = len(source_features)

    means_by_k = {k: np.empty((n_query, n_source), dtype=np.float32) for k in scales}
    stds_by_k = {k: np.empty((n_query, n_source), dtype=np.float32) for k in scales}

    for start in range(0, n_query, batch_size):
        end = min(start + batch_size, n_query)
        _, indices = tree.query(query_xy[start:end], k=requested_k, workers=KNN_WORKERS)

        if requested_k == 1: indices = indices[:, None]
        if exclude_self: indices = indices[:, 1:]

        for k in scales:
            usable_k = min(k, indices.shape[1])
            local_indices = indices[:, :usable_k]
            neighbour_values = reference_values[local_indices]

            means_by_k[k][start:end] = np.nanmean(neighbour_values, axis=1).astype(np.float32)
            stds_by_k[k][start:end] = np.nanstd(neighbour_values, axis=1).astype(np.float32)

    # Dictionary collection to prevent fragmentation warnings
    new_cols_dict = {}
    for k in scales:
        for j, feature in enumerate(source_features):
            new_cols_dict[f"{feature}_nbr{k}_mean"] = means_by_k[k][:, j]
            new_cols_dict[f"{feature}_nbr{k}_std"] = stds_by_k[k][:, j]

    new_cols_df = pd.DataFrame(new_cols_dict, index=query_df.index).replace([np.inf, -np.inf], np.nan).fillna(0)
    return pd.concat([query_df, new_cols_df], axis=1)

def make_model_feature_names(base_feature_names, neighbour_source_features):
    neighbourhood_feature_names = []
    for k in NEIGHBOR_SCALES:
        for feature in neighbour_source_features:
            neighbourhood_feature_names.extend([
                f"{feature}_nbr{k}_mean",
                f"{feature}_nbr{k}_std",
            ])
    return list(base_feature_names) + neighbourhood_feature_names

# Setup spatial groups based on block coordinates
block_x = np.floor(df_madrid["px_key"].to_numpy() / SPATIAL_BLOCK_SIZE).astype(int)
block_y = np.floor(df_madrid["py_key"].to_numpy() / SPATIAL_BLOCK_SIZE).astype(int)
groups = np.char.add(np.char.add(block_x.astype(str), "_"), block_y.astype(str))

source_feats = choose_neighbour_source_features(feature_names)
final_feat_names = make_model_feature_names(feature_names, source_feats)

gkf = GroupKFold(n_splits=N_SPLITS)
fold_scores = []
confusion_matrices = []
oof_true = []
oof_pred = []
oof_proba = []
oof_indices = []

print(f"Starting {N_SPLITS}-Fold Spatial CV on Madrid ({len(final_feat_names)} total features)...")
print("-" * 55)

total_start = perf_counter()
for fold, (train_idx, valid_idx) in enumerate(gkf.split(df_madrid, df_madrid["age_class"], groups), start=1):
    t0 = perf_counter()
    train_df, valid_df = df_madrid.iloc[train_idx].copy(), df_madrid.iloc[valid_idx].copy()

    # Fold-safe neighbourhood extraction
    train_aug = add_neighbourhood_features(train_df, train_df, source_feats, exclude_self=True)
    valid_aug = add_neighbourhood_features(valid_df, train_df, source_feats, exclude_self=False)

    X_train = train_aug[final_feat_names].to_numpy(dtype=np.float32)
    X_valid = valid_aug[final_feat_names].to_numpy(dtype=np.float32)
    y_train = train_df["age_class"].to_numpy(dtype=int)
    y_valid = valid_df["age_class"].to_numpy(dtype=int)

    model = LGBMClassifier(**LGBM_PARAMS)
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    proba = model.predict_proba(X_valid)

    score = f1_score(y_valid, pred, average="macro")
    cm = confusion_matrix(y_valid, pred, labels=[1, 2, 3, 4], normalize="true")

    fold_scores.append(score)
    confusion_matrices.append(cm)
    oof_true.extend(y_valid.tolist())
    oof_pred.extend(pred.tolist())
    oof_proba.extend(proba.tolist())
    oof_indices.extend(valid_idx.tolist())
    
    print(f"  Fold {fold:2d}: macro-F1 = {score:.4f} ({perf_counter() - t0:.1f} s)")

fold_scores = np.asarray(fold_scores, dtype=np.float64)
mean_cm = np.mean(confusion_matrices, axis=0)
oof_true = np.array(oof_true)
oof_pred = np.array(oof_pred)
oof_proba = np.array(oof_proba)
oof_indices = np.array(oof_indices)

print("-" * 55)
print(f"  Mean macro F1: {fold_scores.mean():.4f} +/- {fold_scores.std():.4f}")
print(f"  Total elapsed time: {perf_counter() - total_start:.1f} s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: individual fold scores + mean/std band
ax = axes[0]
xs = np.arange(1, N_SPLITS + 1)
mn = np.mean(fold_scores)
sd = np.std(fold_scores)

ax.scatter(xs, fold_scores, color='steelblue', s=40, zorder=3, alpha=0.8)
ax.axhline(mn, color='navy', linewidth=1.5, linestyle='--', label=f'Mean = {mn:.4f}')
ax.fill_between([0.5, N_SPLITS + 0.5], mn - sd, mn + sd,
                color='steelblue', alpha=0.15, label=f'Std = {sd:.4f}')

ax.set_xlabel('Fold')
ax.set_ylabel('Macro F1')
ax.set_title(f'Madrid: {N_SPLITS}-Fold Spatial CV\nMean={mn:.4f}  Std={sd:.4f}', fontweight='bold')
ax.set_xlim(0.5, N_SPLITS + 0.5)
ax.legend(fontsize=9)
ax.grid(True, linestyle=':', alpha=0.6)

# Right: out-of-fold confusion matrix accumulated over all folds
ax = axes[1]
classes_mad = sorted(np.unique(oof_true))
cm_oof  = confusion_matrix(oof_true, oof_pred, labels=classes_mad)
cm_norm = cm_oof.astype(float) / cm_oof.sum(axis=1, keepdims=True)
xlbls   = [f'C{c}\n{CLASS_LABELS_MADRID[c]}' for c in classes_mad]
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(classes_mad))); ax.set_xticklabels(xlbls, fontsize=8)
ax.set_yticks(range(len(classes_mad))); ax.set_yticklabels(xlbls, fontsize=8)
for i in range(len(classes_mad)):
    for j in range(len(classes_mad)):
        v = cm_norm[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=9,
                color='white' if v > 0.6 else 'black')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('OOF confusion matrix\n(recall per row — Madrid Spatial CV)', fontweight='bold')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

We see that the macro F1 determination is accurate to within roughly plus or minus 0.01. The confusion matrix shows that Classes 2 and 4 acheive higher recall than the other classes.  We also see that Class 2 is over-predicted.  For example, mmost instances of class 1 are predicted as class 2. This is a strong indication that the class imbalance is biasing prediction towards class 2, which is hte largest class.

---

## 4.3a  Spatial Uncertainty Mapping

### `predict_proba` and per-pixel confidence

`LGBMClassifier.predict_proba(X)` returns a **probability vector** $(p_1, p_2, p_3, p_4)$ for each pixel, where $p_k$ is the model's estimated probability for class $k$. These values sum to 1, and the hard label produced by `predict()` is simply $\arg\max_k p_k$.

Using `predict_proba` gives a richer picture than a single label: a pixel with probabilities $(0.90, 0.05, 0.03, 0.02)$ is confidently assigned to class 1, while $(0.30, 0.28, 0.25, 0.17)$ is a near-random guess that happens to land on class 1. The two pixels look identical in a confusion matrix but carry very different levels of certainty.

### Mapping uncertainty across the city

We use `predict_proba` to study the **geographical distribution of uncertainty** — where in the city is the model confident, and where is it unsure? Mapping uncertainty back onto the building grid can reveal:

- Clusters of uncertain pixels along age-class boundaries or in mixed-use districts
- Areas where the domain gap (Madrid→Amsterdam) inflates uncertainty systematically
- Neighbourhoods where targeted labelling would most improve performance

Two complementary measures are shown for each city:

| Measure | Formula | Interpretation |
|---------|---------|----------------|
| **Shannon entropy** | $H = -\sum_k p_k \log_2 p_k$ (bits) | Ranges from 0 (all probability on one class) to 2 bits (uniform over 4 classes). High = uncertain classifier. |
| **KL from true class** | $D = -\log_2 p_{\text{true}}$ (bits) | The KL divergence of the predicted distribution from the true one-hot distribution simplifies to the negative log-probability of the correct class. **0 = model was certain and correct; high = wrong or uncertain.** Colour scale is inverted so bright = accurate. |

In [ ]:
import matplotlib.patches as mpatches

_GREY    = (0.6, 0.6, 0.6)
_NO_BLDG = mpatches.Patch(facecolor=_GREY, edgecolor='darkgrey', label='No building')


def make_pixel_grid(pixel_ids, values, fill_value):
    """Place per-pixel values into a 2-D spatial image using (px_key, py_key) coordinates.
    values: (N,) scalar or (N, C) multi-channel array.
    fill_value: value for grid positions with no building pixel.
    """
    px = pixel_ids[:, 0].astype(int);  px -= px.min()
    py = pixel_ids[:, 1].astype(int);  py -= py.min()
    H, W = py.max() + 1, px.max() + 1
    if values.ndim == 1:
        img = np.full((H, W), fill_value, dtype=np.float32)
        img[py, px] = values
    else:
        img = np.full((H, W, values.shape[1]), fill_value, dtype=np.float32)
        img[py, px] = values
    return img


def spatial_uncertainty_plots(pixel_ids, proba, y_true, title_prefix=''):
    """
    Set 1 (grey background):
        Left : RGB image where R=p2*256, G=p3*256, B=p4*256
        Right: actual class colours  (C1=black, C2=red, C3=green, C4=blue)
    Set 2 (hot_r — bright=good, dark=bad):
        Left : Shannon entropy in bits  (bright=certain, dark=uncertain)
        Right: KL(true||pred) = -log2(p_true_class)  (bright=accurate, dark=inaccurate)
    """
    eps = 1e-10

    # ── Set 1: colour images ──────────────────────────────────────────────────
    rgb_vals = (np.floor(proba[:, 1:] * 256).clip(0, 255) / 255.0).astype(np.float32)
    rgb_img  = make_pixel_grid(pixel_ids, rgb_vals, fill_value=0.70)

    _class_rgb = np.array([[0,0,0],[1,0,0],[0,1,0],[0,0,1]], dtype=np.float32)
    cls_img    = make_pixel_grid(pixel_ids, _class_rgb[y_true - 1], fill_value=0.7)

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(rgb_img);  axes[0].axis('off')
    axes[0].set_title(f'{title_prefix}\nPredicted probabilities  '
                      r'(R=$p_2$, G=$p_3$, B=$p_4$)', fontsize=10)

    axes[1].imshow(cls_img);  axes[1].axis('off')
    axes[1].set_title(f'{title_prefix}\nActual classes', fontsize=10)
    axes[1].legend(handles=[
        mpatches.Patch(color='black', label='C1'),
        mpatches.Patch(color='red',   label='C2'),
        mpatches.Patch(color='green', label='C3'),
        mpatches.Patch(color='blue',  label='C4'),
        _NO_BLDG,
    ], loc='lower right', fontsize=8, framealpha=0.85)

    plt.suptitle('Set 1 — class colour maps  (grey = no building pixel)',
                 fontweight='bold')
    plt.tight_layout()
    plt.show()

    # ── Set 2: uncertainty / accuracy maps ────────────────────────────────────
    H_vals           = -(proba * np.log2(proba + eps)).sum(axis=1)
    true_class_proba = proba[np.arange(len(y_true)), y_true - 1]
    kl_vals          = -np.log2(true_class_proba + eps)

    H_img  = make_pixel_grid(pixel_ids, H_vals.astype(np.float32),  fill_value=np.nan)
    kl_img = make_pixel_grid(pixel_ids, kl_vals.astype(np.float32), fill_value=np.nan)

    cmap_h  = plt.cm.hot_r.copy(); cmap_h.set_bad(_GREY)
    cmap_kl = plt.cm.hot_r.copy(); cmap_kl.set_bad(_GREY)

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    im0 = axes[0].imshow(H_img, cmap=cmap_h, vmin=0, vmax=2)
    axes[0].axis('off')
    axes[0].set_title(f'{title_prefix}\nShannon entropy  '
                      '(bright=certain, dark=uncertain  |  max=2 bits)', fontsize=10)
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(kl_img, cmap=cmap_kl, vmin=0, vmax=2)
    axes[1].axis('off')
    axes[1].set_title(f'{title_prefix}\n'
                      r'KL from true class $= -\log_2 p_{\rm true}$ (bits)'
                      '\nbright=accurate, dark=inaccurate', fontsize=10)
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    plt.suptitle('Set 2 — uncertainty & accuracy maps  '
                 '(grey=no building  |  bright=good, dark=bad)', fontweight='bold')
    plt.tight_layout()
    plt.show()


def spatial_accuracy_plots(pixel_ids, y_true, y_pred, title_prefix=''):
    """
    Plot 1: black=correct, red=incorrect.
    Plot 2: C1=black, C2=mid-dark red, C3=mid-dark green, C4=mid-dark blue, incorrect=yellow.
    Grey background on both.
    """
    correct = (y_true == y_pred)

    p1_vals = np.where(
        correct[:, None],
        np.array([[0, 0, 0]], dtype=np.float32),
        np.array([[1, 0, 0]], dtype=np.float32),
    )
    p1_img = make_pixel_grid(pixel_ids, p1_vals, fill_value=0.6)

    correct_colors = np.array([
        [  0,   0,   0],
        [140,   0,   0],
        [  0, 140,   0],
        [  0,   0, 140],
    ], dtype=np.float32) / 255.0
    yellow  = np.array([[1, 1, 0]], dtype=np.float32)
    p2_vals = np.where(correct[:, None], correct_colors[y_true - 1], yellow)
    p2_img  = make_pixel_grid(pixel_ids, p2_vals.astype(np.float32), fill_value=0.6)

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    axes[0].imshow(p1_img);  axes[0].axis('off')
    axes[0].set_title(f'{title_prefix}\nCorrect vs incorrect', fontsize=10)
    axes[0].legend(handles=[
        mpatches.Patch(color='black', label='Correct'),
        mpatches.Patch(color='red',   label='Incorrect'),
        _NO_BLDG,
    ], loc='lower right', fontsize=8, framealpha=0.85)

    axes[1].imshow(p2_img);  axes[1].axis('off')
    axes[1].set_title(f'{title_prefix}\nCorrect class  /  incorrect', fontsize=10)
    axes[1].legend(handles=[
        mpatches.Patch(color='black',                      label='C1 correct'),
        mpatches.Patch(color=(140/255, 0,       0      ),  label='C2 correct'),
        mpatches.Patch(color=(0,       140/255, 0      ),  label='C3 correct'),
        mpatches.Patch(color=(0,       0,       140/255),  label='C4 correct'),
        mpatches.Patch(color='yellow',                     label='Incorrect'),
        _NO_BLDG,
    ], loc='lower right', fontsize=8, framealpha=0.85)

    plt.suptitle('Correct / incorrect pixel maps  (grey = no building pixel)',
                 fontweight='bold')
    plt.tight_layout()
    plt.show()


# ── Madrid OOF ────────────────────────────────────────────────────────────────
pixel_ids_oof = pixel_ids_madrid[oof_indices]
spatial_uncertainty_plots(pixel_ids_oof, oof_proba, oof_true,
                          title_prefix='Madrid CV (out-of-fold)')
spatial_accuracy_plots(pixel_ids_oof, oof_true, oof_pred,
                       title_prefix='Madrid CV (out-of-fold)')


We can see that despite the relatively low accuracy rates, the algorithm seems to be correctly identifying the major local areas where each class dominates. this suggests that a subsequent ML algorithm that acts on the soft-probabiliy output image may acheive improved accuracy.

---

## 4.4 Final Model: Train on All Madrid Data

After cross-validation has confirmed the performance estimate, we train a final model on
the complete Madrid dataset. This maximises the information available and produces the
model used for Amsterdam transfer.

In this notebook we have not taken class imbalance into account.  When class imbalance is present, larger classes tend to be overpredicted, because a randomly-chosen datapoint has a larger probability of being in the larger class. in your model you may wish to take this into account.

**NOTE**: In this notebook, we use random forest as a baseline, with no parameter optimization. Your job is to improve on this baseline.  Be creative--try other techniques.

In [ ]:
print("TRAINING FINAL STAGE-1 MODEL ON ALL MADRID PIXELS\n")

full_aug = add_neighbourhood_features(df_madrid, df_madrid, source_feats, exclude_self=True)
X_full = full_aug[final_feat_names].to_numpy(dtype=np.float32)
y_full = df_madrid["age_class"].to_numpy(dtype=int)

final_model = LGBMClassifier(**LGBM_PARAMS)
final_model.fit(X_full, y_full)

# Feature Importance
importance = final_model.feature_importances_
importance_df = pd.DataFrame({"feature": final_feat_names, "importance": importance})
importance_df = importance_df.sort_values("importance", ascending=False).reset_index(drop=True)

print("=" * 70)
print("TOP 25 FINAL-MODEL FEATURES")
print("=" * 70)
print(importance_df.head(25).to_string(index=False))

# Save Package
OUTPUT_DIR = Path("../models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = OUTPUT_DIR / "stage1_madrid_lightgbm.pkl"

package = {
    "model": final_model,
    "base_feature_names": list(feature_names),
    "neighbour_source_features": list(source_feats),
    "neighbour_scales": list(NEIGHBOR_SCALES),
    "final_feature_names": list(final_feat_names),
    "madrid_reference_coords": df_madrid[["px_key", "py_key"]].to_numpy(dtype=np.float64),
    "madrid_reference_base_features": df_madrid[source_feats].to_numpy(dtype=np.float32),
    "scaler": data.get("scaler"),
    "feat_groups": data.get("feat_groups"),
    "spatial_block_size": SPATIAL_BLOCK_SIZE,
    "lightgbm_params": LGBM_PARAMS,
}

with open(MODEL_PATH, "wb") as f:
    pickle.dump(package, f)

print(f"\nSaved final Stage-1 package to:\n  {MODEL_PATH}")

---

## 4.5 Feature Importance

Random Forest feature importance (mean decrease in impurity) shows which temporal
statistics are most useful for distinguishing age classes. Temporal features should
rank highly if the time-series approach is capturing meaningful signals.

In [ ]:
import matplotlib.pyplot as plt

importances = final_model.feature_importances_
sorted_idx  = np.argsort(importances)

TOP_N = 20
plt.figure(figsize=(10, 6))
plt.barh(range(TOP_N), importances[sorted_idx[:TOP_N]], color='steelblue')
plt.yticks(range(TOP_N), [final_feat_names[i] for i in sorted_idx[:TOP_N]], fontsize=9)
plt.xlabel("LightGBM Feature Importance", fontweight='bold')
plt.title(f"Top {TOP_N} Spatial-Temporal Features (Madrid Model)", fontweight='bold')
plt.tight_layout()
plt.show()

We see that a large number of features are nearly equal in importance. This is no doubt at least partially due to the high correlation between features.   But it also indicates that the Random Forest may be able to discern differencfes based on complex relationships, that would be difficult for a human to discern.

---

## 4.6 Zero-Shot Transfer to Amsterdam

We apply the Madrid-trained model to all Amsterdam pixels without using any Amsterdam labels.
This establishes the **baseline domain gap**: how much performance we lose when moving to
a city the model has never seen.

In [ ]:
from sklearn.metrics import f1_score, classification_report

print("Augmenting Amsterdam pixels with spatial neighborhood features for zero-shot transfer...")
df_amsterdam_aug = add_neighbourhood_features(
    query_df=df_amsterdam, 
    reference_df=df_amsterdam, 
    source_features=source_feats, 
    exclude_self=True
)

X_amsterdam_aug = df_amsterdam_aug[final_feat_names].to_numpy(dtype=np.float32)

y_ams_pred_zero = final_model.predict(X_amsterdam_aug)

f1_zero = f1_score(y_amsterdam, y_ams_pred_zero, average='macro')
print('Zero-shot transfer to Amsterdam (no Amsterdam labels):')
print(f'  Macro F1: {f1_zero:.4f}')
print()

classes_ams = sorted(np.unique(y_amsterdam))
print(classification_report(
    y_amsterdam, y_ams_pred_zero,
    labels=classes_ams,
    target_names=[f'Class {c}' for c in classes_ams],
    digits=4,
))

In [ ]:
cm_zero = confusion_matrix(y_amsterdam, y_ams_pred_zero,
                           labels=sorted(np.unique(y_amsterdam)))
cm_norm_zero = cm_zero.astype(float) / cm_zero.sum(axis=1, keepdims=True)
xlbls_ams = [f'C{c}\n{CLASS_LABELS_AMSTERDAM[c]}' for c in sorted(np.unique(y_amsterdam))]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_norm_zero, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(xlbls_ams))); ax.set_xticklabels(xlbls_ams, fontsize=8)
ax.set_yticks(range(len(xlbls_ams))); ax.set_yticklabels(xlbls_ams, fontsize=8)
for i in range(len(xlbls_ams)):
    for j in range(len(xlbls_ams)):
        v = cm_norm_zero[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=10,
                color='white' if v > 0.6 else 'black')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Zero-shot transfer — Amsterdam\nRecall per class  (macro F1={f1_zero:.3f})',
             fontweight='bold')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


Accuracy is poor for the zero-shot model.  this is to be expected, because buildings in Amsterdam have quite different characteristics from buildings in Madrid. The purpose of the few-shot retraining is to identify Amsterdam-specific clusters within the same metric space. The Amsterdam pixels are then assigned to these recalculated clusters. 

In [ ]:
# Safeguard to ensure pixel_ids_amsterdam is defined
if 'pixel_ids_amsterdam' not in locals():
    pixel_ids_amsterdam = data.get('pixel_ids_amsterdam', coords_amsterdam)

# Ensure Amsterdam is spatially augmented if not already done
if 'X_amsterdam_eval' not in locals():
    df_amsterdam_aug = add_neighbourhood_features(
        query_df=df_amsterdam, 
        reference_df=df_amsterdam, 
        source_features=source_feats, 
        exclude_self=True
    )
    X_amsterdam_eval = df_amsterdam_aug[final_feat_names].to_numpy(dtype=np.float32)

# Amsterdam zero-shot — predict_proba from the LightGBM final model
proba_amsterdam_zero = final_model.predict_proba(X_amsterdam_eval)

spatial_uncertainty_plots(pixel_ids_amsterdam, proba_amsterdam_zero, y_amsterdam,
                          title_prefix='Amsterdam zero-shot')
spatial_accuracy_plots(pixel_ids_amsterdam, y_amsterdam, y_ams_pred_zero,
                       title_prefix='Amsterdam zero-shot')

The graphs show clearly that Class 2 is being wildly overpredicted in the zero-shot model.

---

## 4.7 Few-Shot Prototype Transfer

The prototype classifier is the inference step of **prototype networks**, a standard metric learning technique:

1. For each class, compute the **prototype** = mean feature vector of the labelled examples in Amsterdam.
2. Classify unlabelled points by assigning them to the nearest prototype (Euclidean distance).

This is equivalent to 1-NN where the single "neighbour" is a smoothed class representative. It generalises better than raw k-NN with very few labels because averaging reduces noise.

We run the experiment for several label budgets (samples per class) and repeat each 10 times with different random draws to estimate variance. The resulting plot shows performance as a function of label budget — the key question for deciding how much annotation effort Amsterdam requires.

In [ ]:
def prototype_predict(X_support, y_support, X_query):
    classes = np.unique(y_support)
    prototypes = np.stack([X_support[y_support == c].mean(axis=0) for c in classes])
    dists = np.linalg.norm(X_query[:, None, :] - prototypes[None, :, :], axis=2)
    return classes[dists.argmin(axis=1)]

SHOTS_PER_CLASS = [5, 10, 25, 50, 100, 200]
N_TRIALS = 10
rng = np.random.default_rng(42)
classes_ams = np.unique(y_amsterdam)

proto_results = {n: [] for n in SHOTS_PER_CLASS}

# Ensure we use the spatially augmented Amsterdam feature matrix
X_amsterdam_eval = df_amsterdam_aug[final_feat_names].to_numpy(dtype=np.float32)

print("Running Few-Shot Prototype Transfer on Amsterdam (Spatial Feature Space)...")
print("-" * 55)

for n_shots in SHOTS_PER_CLASS:
    for _ in range(N_TRIALS):
        support_idx = []
        for cls in classes_ams:
            cls_idx = np.where(y_amsterdam == cls)[0]
            chosen = rng.choice(cls_idx,
                                min(n_shots, len(cls_idx)),
                                replace=False)
            support_idx.extend(chosen.tolist())

        support_idx = np.array(support_idx)
        query_mask = np.ones(len(y_amsterdam), dtype=bool)
        query_mask[support_idx] = False

        y_pred = prototype_predict(
            X_amsterdam_eval[support_idx], y_amsterdam[support_idx],
            X_amsterdam_eval[query_mask]
        )
        f1 = f1_score(y_amsterdam[query_mask], y_pred,
                      average='macro', zero_division=0)
        proto_results[n_shots].append(f1)

    mn = np.mean(proto_results[n_shots])
    std = np.std(proto_results[n_shots])
    print(f"  {n_shots:4d} shots/class: macro-F1 = {mn:.4f} +/- {std:.4f}")

print("-" * 55)

In [ ]:
import matplotlib.pyplot as plt

# Bridge variable names from our 10-fold CV script
fold_f1 = fold_scores

fig, ax = plt.subplots(figsize=(10, 5))

ns    = SHOTS_PER_CLASS
means = [np.mean(proto_results[n]) for n in ns]
stds  = [np.std(proto_results[n])  for n in ns]

# Individual trial scatter
for n in ns:
    ax.scatter([n] * N_TRIALS, proto_results[n],
               color='#fd8d3c', alpha=0.35, s=20, zorder=3)

# Mean + error band
ax.errorbar(ns, means, yerr=stds, fmt='o-', color='#fd8d3c',
            linewidth=2, markersize=7, capsize=4,
            label='Prototype transfer (Amsterdam, mean +/- std)')

# Baselines
ax.axhline(f1_zero, color='steelblue', linestyle='--', linewidth=1.5,
           label=f'Zero-shot (no Amsterdam labels): {f1_zero:.3f}')
ax.axhline(np.mean(fold_f1), color='green', linestyle='--', linewidth=1.5,
           label=f'Madrid CV: {np.mean(fold_f1):.3f} +/- {np.std(fold_f1):.3f}')

ax.set_xlabel('Labeled Amsterdam examples per class', fontweight='bold')
ax.set_ylabel('Macro F1', fontweight='bold')
ax.set_xscale('log')
ax.set_title('Few-shot prototype transfer: Amsterdam F1 vs. label budget',
             fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

The graph  shows that even with 5 examples per class, a huge boost in accuracy is seen compared to the zero-shot model. As few as 25 samples per class increases by an additional 10%, and 100 samples per class is sufficient to clearly establish the class centers, with additional samples bringing only marginal improvement.

In [ ]:

# Ensure Amsterdam tick labels are mapped correctly
classes_ams = sorted(np.unique(y_amsterdam))
xlbls_ams = [f'C{c}\n{CLASS_LABELS_AMSTERDAM[c]}' for c in classes_ams]

# Use the spatially augmented Amsterdam feature space
X_amsterdam_eval = df_amsterdam_aug[final_feat_names].to_numpy(dtype=np.float32)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
rng_cm = np.random.default_rng(42)

for ax, n_shots in zip(axes, [25, 100]):
    support_idx = []
    for cls in classes_ams:
        cls_idx = np.where(y_amsterdam == cls)[0]
        chosen  = rng_cm.choice(cls_idx, min(n_shots, len(cls_idx)), replace=False)
        support_idx.extend(chosen.tolist())

    support_idx = np.array(support_idx)
    query_mask  = np.ones(len(y_amsterdam), dtype=bool)
    query_mask[support_idx] = False

    y_pred_shot = prototype_predict(
        X_amsterdam_eval[support_idx], y_amsterdam[support_idx],
        X_amsterdam_eval[query_mask]
    )
    y_true_q = y_amsterdam[query_mask]

    cm_shot  = confusion_matrix(y_true_q, y_pred_shot, labels=classes_ams)
    cm_snorm = cm_shot.astype(float) / cm_shot.sum(axis=1, keepdims=True)
    f1_shot  = f1_score(y_true_q, y_pred_shot, average='macro', zero_division=0)

    im = ax.imshow(cm_snorm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(xlbls_ams))); ax.set_xticklabels(xlbls_ams, fontsize=8)
    ax.set_yticks(range(len(xlbls_ams))); ax.set_yticklabels(xlbls_ams, fontsize=8)
    for i in range(len(xlbls_ams)):
        for j in range(len(xlbls_ams)):
            v = cm_snorm[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=10,
                    color='white' if v > 0.6 else 'black')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'Prototype transfer: {n_shots} shots/class\n'
                 f'Amsterdam  (macro F1={f1_shot:.3f})', fontweight='bold')
    plt.colorbar(im, ax=ax)

plt.suptitle('Few-shot confusion matrices: recall per row', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

The recall performance for Amsterdam just about matches (and for some classes exceeds!) the recall obtained for Madrid.

In [ ]:
def prototype_predict_proba(X_support, y_support, X_query):
    """Soft class probabilities via softmax of negative L2 distances to prototypes."""
    classes = np.unique(y_support)
    prototypes = np.stack([X_support[y_support == c].mean(axis=0) for c in classes])
    dists = np.linalg.norm(X_query[:, None, :] - prototypes[None, :, :], axis=2)
    neg_d = -dists
    neg_d -= neg_d.max(axis=1, keepdims=True)
    exp_d = np.exp(neg_d)
    return classes, exp_d / exp_d.sum(axis=1, keepdims=True)

# Safeguard to ensure pixel_ids_amsterdam is defined
if 'pixel_ids_amsterdam' not in locals():
    pixel_ids_amsterdam = data.get('pixel_ids_amsterdam', coords_amsterdam)

# Use the spatially augmented Amsterdam feature space
X_amsterdam_eval = df_amsterdam_aug[final_feat_names].to_numpy(dtype=np.float32)

# Reproduce the exact 25-shot split using the spatial feature matrix
rng_25 = np.random.default_rng(42)
n_shots_25 = 25
support_idx_25 = []
for cls in classes_ams:
    cls_idx = np.where(y_amsterdam == cls)[0]
    chosen = rng_25.choice(cls_idx, min(n_shots_25, len(cls_idx)), replace=False)
    support_idx_25.extend(chosen.tolist())
support_idx_25 = np.array(support_idx_25)
query_mask_25 = np.ones(len(y_amsterdam), dtype=bool)
query_mask_25[support_idx_25] = False

X_sup_25 = X_amsterdam_eval[support_idx_25]
y_sup_25 = y_amsterdam[support_idx_25]
X_qry_25 = X_amsterdam_eval[query_mask_25]
y_qry_25 = y_amsterdam[query_mask_25]
pids_qry_25 = pixel_ids_amsterdam[query_mask_25]

_, proba_25 = prototype_predict_proba(X_sup_25, y_sup_25, X_qry_25)
y_pred_25 = prototype_predict(X_sup_25, y_sup_25, X_qry_25)

spatial_uncertainty_plots(pids_qry_25, proba_25, y_qry_25,
                          title_prefix='Amsterdam 25-shot prototype (query pixels)')
spatial_accuracy_plots(pids_qry_25, y_qry_25, y_pred_25,
                       title_prefix='Amsterdam 25-shot prototype (query pixels)')

As with Madrid, the model is identifying most local areas where each specific classes predominates. 

---

## 4.8 Summary

**Madrid cross-validation** (5-fold, with error bars) gives the headline performance
estimate. Error bars reveal how stable the estimate is — tight bars mean the result
is reliable; wide bars mean more folds or data are needed.

**Zero-shot transfer** to Amsterdam shows the raw domain gap: how much performance
degrades when moving to a city the model has never seen. A large gap (>0.1 macro F1)
suggests the feature space does not transfer perfectly, which motivates the few-shot step.

**Few-shot prototype transfer** closes the gap progressively as more Amsterdam labels
are added. The prototype classifier is the simplest form of metric learning inference:
- With very few labels (5-10 per class) it may already improve over zero-shot.
- With 100-200 labels per class the gap to the Madrid CV score should largely close.

### Ideas for improvement

(Note these ideas are generated by Claude, and are not guaranteed.  I didn't check them carefully.  Do your own brainstorming!)

**Better feature representation (higher leverage)**
- Per-decade statistics instead of two fixed periods
- Slope of linear trend per band (captures systematic change over time)
- Year of maximum/minimum reflectance per band
- Changepoint detection features: year of largest single-year jump

**Better metric learning**
- Learn an explicit embedding with triplet loss (neural network)
- Use the Madrid-trained RF leaf node assignments as a kernel embedding
  (`rf.apply()` returns leaf IDs; co-occurrence rate is a natural similarity)
- Domain-adversarial training: penalise features that predict city identity

**Better few-shot transfer**
- Fine-tune on Amsterdam labels rather than using prototypes
- Use the Amsterdam unlabelled data for self-training (pseudo-labels from zero-shot)